# Machine Learning Challenge (Individual Assignment, 30%)


<div class="alert-danger">

**Your name:** [Enter your name here] 

</div>

## Introduction

Develop a model to predict the numerical target `Y` from anonymous, synthetic features. Your test predictions are evaluated using **mean absolute error (MAE)**; lower is better.

Read [ReadMe.md](ReadMe.md) for the full assignment rules, grading criteria, file descriptions, and required deliverables. **Submit by 15 December 2026 via [Canvas](https://canvas.esmt.berlin/courses/794/assignments/11053).**

Use this notebook to develop and document your solution. The cells below provide data-loading helpers, baseline forecasts, and submission generation.


## Preliminaries
Imports of standard libraries are included below. Add any further imports below and use `SEED` for reproducible random operations.

Keep `train_X.csv`, `train_y.csv`, and `test_X.csv` in the same folder as this notebook, or adjust `DATA_DIR` before running the helper definitions.

- `load_training_data()` returns `X` and `y`, aligned by `ID`; IDs are excluded from the model features.
- `load_test_data()` returns test features with IDs in the index.
- `write_submission(test_ids, predictions, filename)` writes columns `ID` and `Y`, preserving the supplied ID/prediction order.


In [ ]:
# Standard library imports
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression

# Add more imports if needed. You can use any Python library and ML algorithm

SEED = 67
DATA_DIR = Path('.') # adjust if you store the data elsewhere

In [ ]:
def load_training_data(data_dir=DATA_DIR):
    """Return training features X and targets y, aligned by case ID."""
    data_dir = Path(data_dir)
    # ID becomes the row index and is excluded from the model's feature columns.
    X = pd.read_csv(data_dir / 'train_X.csv', index_col='ID')
    targets = pd.read_csv(data_dir / 'train_y.csv', index_col='ID')
    # Match target rows to feature IDs even if the label file has been reordered.
    y = targets.loc[X.index, 'Y']
    return X, y

In [ ]:
def load_test_data(data_dir=DATA_DIR):
    """Return unlabeled test features, retaining case IDs as the row index."""
    return pd.read_csv(Path(data_dir) / 'test_X.csv', index_col='ID')

In [ ]:
def write_submission(test_ids, predictions, filename='submission.csv'):
    """Write ID and target Y to a CSV; predictions follow the supplied ID order."""
    ids = pd.Index(test_ids, name='ID')
    # An array prevents pandas from silently aligning a prediction Series by index.
    predictions = np.asarray(predictions, dtype=float)
    if predictions.shape != (len(ids),):
        raise ValueError('Provide one prediction per test ID as a one-dimensional array or Series.')
    if ids.has_duplicates or ids.isna().any():
        raise ValueError('Test IDs must be unique and non-missing.')
    if not np.isfinite(predictions).all():
        raise ValueError('Predictions must contain only finite numerical values.')

    submission = pd.DataFrame({'Y': predictions}, index=ids)
    output_path = Path(filename)
    # Write the index exactly once as the ID column, followed by target column Y.
    submission.to_csv(output_path, index_label='ID')
    return output_path

## Your solution

Add code and Markdown cells below to develop and explain your approach. Compare models using validation data drawn from the training sample; fit learned preprocessing only on the relevant training rows. Keep features and targets aligned.

The starter code compares linear regression with a constant forecast equal to the target mean. Its **in-sample MAEs** illustrate training fit; use validation to assess performance on unseen cases.

After selecting your procedure, fit it on the available training data and assign the fitted predictor to `final_model`. The baseline already uses this name, so you can run the submission cell immediately.

Do not forget that any preprocessing of the training data must be performed on the test data before you can apply for final model.


In [ ]:
# Load the training data
X, y = load_training_data()

# Fit linear regression and evaluate its predictions in-sample
final_model = LinearRegression()
final_model.fit(X, y)
training_predictions = final_model.predict(X)
regression_mae = mean_absolute_error(y, training_predictions)

# Compare against a constant forecast equal to the target's mean (over training data)
target_mean = y.mean()
mean_predictions = np.full(len(y), target_mean)
mean_mae = mean_absolute_error(y, mean_predictions)

print(f'Training target mean: {target_mean:.3f}')
print(f'Mean forecast MAE (in-sample): {mean_mae:.3f}')
print(f'Linear regression MAE (in-sample): {regression_mae:.3f}')


### Time to develop your solution

In [ ]:
# Your code goes here ...


## Submission

Run the cell below to create `submission.csv`. Your `final_model` must apply any fitted preprocessing when predicting the test features.

Check that the CSV contains columns `ID` and `Y` and one finite prediction for every test ID, in test-row order. See [sample_submission.csv](sample_submission.csv) for the format; its values and ellipsis are placeholders.

Before submitting, restart the kernel and run all cells in order. Confirm that the notebook reproduces your CSV, retain your explanations and outputs, and include the other deliverables listed in [ReadMe.md](ReadMe.md).


In [ ]:
# Load the test data
X_test = load_test_data()

# Recall that the test data is loaded in raw format.
# Your need to apply the same preprocessing steps to the test data as you did for the training data.


# Assert that the relevant variables are available
assert 'final_model' in globals(), "You need to train a model and store it in the variable 'final_model'."
assert 'X_test' in globals(), "You need to load the test data and store it in the variable 'X_test'."
test_predictions = final_model.predict(X_test)
assert test_predictions.shape == (len(X_test),), "The predictions must be a one-dimensional array with the same length as the test data."
submission_path = write_submission(X_test.index, test_predictions, 'submission.csv')
print(submission_path)

print('Next step is uploading your solution to Canvas.')
print('🍀 Good luck 🍀')